### Parsing the full log data to structure it with file names as Y-axis and categories as X-axis


In [5]:
import pandas as pd

In [6]:
full_log_entries = """
log2111:insertion_time_ns = 19488512526
log2111:rd_time_ns = 5895754
log2111:total_cpu_time_ns = 3987000.00
log2111:whole block total_cpu_time_ns = 12999000.00
log2111:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 8617191043
log2111:fixed #PQ = 5000 point_query_time_on_currently_deleted_all_ns = 1978496039
log2111:all_time_ns = 260384278556
log2112:insertion_time_ns = 19510919633
log2112:rd_time_ns = 5895767
log2112:total_cpu_time_ns = 6999000.00
log2112:whole block total_cpu_time_ns = 12998000.00
log2112:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 8077058837
log2112:fixed #PQ = 5000 point_query_time_on_currently_deleted_all_ns = 1932765469
log2112:all_time_ns = 256505913458
log2113:insertion_time_ns = 19827097324
log2113:rd_time_ns = 5841043
log2113:total_cpu_time_ns = 4000000.00
log2113:whole block total_cpu_time_ns = 12998000.00
log2113:fixed #PQ = 5000 point_query_time_on_existing_keys_ns = 7901032710
log2113:fixed #PQ = 5000 point_query_time_on_currently_deleted_all_ns = 1516883270
log2113:all_time_ns = 255062027861
"""

In [7]:

# Creating a dictionary to store structured data
structured_data = {}

# Process each log entry
for line in full_log_entries.strip().split("\n"):
    parts = line.split(" = ")
    if len(parts) == 2:
        log_id, metric = parts[0].split(":", 1)
        value = float(parts[1]) if "." in parts[1] else int(parts[1])
        
        if log_id not in structured_data:
            structured_data[log_id] = {}
        
        structured_data[log_id][metric] = value
    
    if len(parts) == 3:
        log_id, metric_1 = parts[0].split(":", 1)
        metric = metric_1 + " = " + parts[1]
        # print(log_id, metric)
        value = float(parts[2]) if "." in parts[2] else int(parts[2])
        
        if log_id not in structured_data:
            structured_data[log_id] = {}
        
        structured_data[log_id][metric] = value


In [8]:

# Define the required order based on the 4-digit prefix after "log"
required_order = [11]
# required_order = [11, 21, 31, 41, 51, 61, 71, 81, 91, 121, 131, 141, 151, 101, 111, 161, 171]
offset = 4000
n_group = 3

# print(str(list(structured_data.keys())))
filenames = list(structured_data.keys())

# Reinitialize a dictionary for ordered storage based on the correct mapping
sorted_filenames_corrected = {key: [] for key in required_order}

# Process each filename and categorize based on the given order
for filename in filenames:
    # num_part = (int(filename[3:7]) % 10)
    num_part = (int(filename[3:7]) - offset) // 100 * 10 + (int(filename[3:7]) % 100) // 10
    if num_part in sorted_filenames_corrected:
        sorted_filenames_corrected[num_part].append(filename)

# sorted_filenames_corrected = 
file_names_list_2d = [list(val) for val in sorted_filenames_corrected.values()]
file_names_list_2d = list(zip(*file_names_list_2d))  # Correct transposition



print(file_names_list_2d)
# # Ensure ordering is correct by maintaining the specified order
# df_sorted_filenames = pd.DataFrame.from_dict(sorted_filenames_corrected, orient="index").T

# # Display the corrected sorted groups
# tools.display_dataframe_to_user(name="Reordered Log Groups (Final)", dataframe=df_sorted_filenames)



[]


In [9]:

# Convert dictionary to DataFrame
df_structured = pd.DataFrame.from_dict(structured_data, orient="index")

# Reset index and rename columns
df_structured.reset_index(inplace=True)
df_structured.rename(columns={"index": "File Name"}, inplace=True)



print(df_structured["File Name"])

0    log2111
1    log2112
2    log2113
Name: File Name, dtype: object


In [12]:
# # Initialize a dictionary to store the new structured data
# new_df_table = {}

# # Iterate through each group ID in sorted order
# for file_group in file_names_list_2d:
#     for log_id in file_group:
#         # Extract the corresponding row from df_structured
#         line = df_structured[df_structured["File Name"] == log_id]
#         # print(log_id)
#         # Store it in the new table
#         new_df_table[log_id] = line

# # Convert dictionary to a structured DataFrame
# df_new_table = pd.concat(new_df_table.values())
df_new_table = df_structured

# Save the new structured table
structured_table_path = "./structured_log_data.xlsx"
df_new_table.to_excel(structured_table_path, index=False)

print(df_new_table)
# # Provide download link
# structured_table_path


  File Name  insertion_time_ns  rd_time_ns  total_cpu_time_ns   
0   log2111        19488512526     5895754          3987000.0  \
1   log2112        19510919633     5895767          6999000.0   
2   log2113        19827097324     5841043          4000000.0   

   whole block total_cpu_time_ns   
0                     12999000.0  \
1                     12998000.0   
2                     12998000.0   

   fixed #PQ = 5000 point_query_time_on_existing_keys_ns   
0                                         8617191043      \
1                                         8077058837       
2                                         7901032710       

   fixed #PQ = 5000 point_query_time_on_currently_deleted_all_ns   all_time_ns  
0                                         1978496039              260384278556  
1                                         1932765469              256505913458  
2                                         1516883270              255062027861  


In [11]:


# display(df_structured)

# # Save structured data to Excel
# #structured_excel_filename = "/mnt/data/structured_log_data.xlsx"
# structured_excel_filename = "./structured_log_data.xlsx"
# # structured_excel_filename = "./structured_log_data.csv"
# df_structured.to_excel(structured_excel_filename, index=False)

# ## Provide download link
# #structured_excel_filename

